# 🧠 Brain Tumor Segmentation — BraTS 2023 GLI

**Notebook huấn luyện 3D U-Net phân đoạn khối u não**

| Thông tin | Chi tiết |
|---|---|
| **Dataset** | BraTS 2023 GLI Challenge — Training Data |
| **Model** | 3D U-Net (MONAI) |
| **Framework** | PyTorch + MONAI |
| **Input** | 4 MRI modalities: T1n, T1c, T2w, T2f — shape `(4, 240, 240, 155)` |
| **Output** | Segmentation mask — 4 classes: Background / NCR / ED / ET |
| **Patch size** | 128 × 128 × 128 |

---

## 📋 Nội dung Notebook

1. Cài đặt thư viện
2. Imports & Config tập trung
3. Cấu hình đường dẫn dữ liệu
4. Khảo sát dữ liệu (EDA)
5. Dataset & DataLoader
6. Định nghĩa mô hình 3D U-Net
7. Training Loop (AMP + tqdm + Checkpoint)
8. Đồ thị Learning Curves
9. Đánh giá cuối cùng (Per-class Dice)
10. Export model (TorchScript & ONNX)

## 1. Cài Đặt Thư Viện

In [ ]:
# Cài đặt MONAI và các thư viện cần thiết
!pip install monai nibabel scikit-learn tqdm -q
print("✅ Cài đặt hoàn tất!")

## 2. Imports & Config Tập Trung

> **Tất cả imports và hyperparameters được gom về 1 chỗ duy nhất.**  
> Khi muốn thay đổi tham số, chỉ cần chỉnh tại `Config` — không cần tìm khắp notebook.

In [ ]:
# ============================================================
#  IMPORTS
# ============================================================
import os
import sys
import time
import json
import random
import logging
import hashlib
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import defaultdict
from typing import Dict, List, Optional

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

import monai
import monai.transforms as mt
from monai.networks.nets import UNet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from monai.data import decollate_batch
from monai.transforms import AsDiscrete

# ============================================================
#  LOGGING SETUP
# ============================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger("BraTS")

# ============================================================
#  CONFIG — Tất cả hyperparameters ở một chỗ
# ============================================================
@dataclass
class Config:
    # --- Reproducibility ---
    seed: int = 42

    # --- Data ---
    val_split: float = 0.15
    modalities: tuple = ("t1n", "t1c", "t2w", "t2f")

    # --- Model ---
    in_channels: int = 4           # T1n, T1c, T2w, T2f
    out_channels: int = 4          # Background, NCR, ED, ET
    patch_size: tuple = (128, 128, 128)
    unet_features: tuple = (32, 64, 128, 256, 512)
    unet_strides: tuple = (2, 2, 2, 2)
    unet_res_units: int = 2

    # --- Training ---
    max_epochs: int = 100
    lr: float = 2e-4
    weight_decay: float = 1e-5
    grad_clip: float = 1.0         # gradient clipping max norm
    val_interval: int = 2          # validate mỗi N epoch
    sw_batch_size: int = 2         # sliding window batch size

    # --- DataLoader ---
    batch_size: int = 1
    num_workers: int = 2
    num_samples: int = 2           # RandCropByPosNegLabeld

    # --- Paths ---
    ckpt_dir: str = "checkpoints"
    models_dir: str = "models"

CFG = Config()

# ── Class names & label mapping ──────────────────────────────
CLASS_NAMES  = {0: "Background", 1: "NCR", 2: "ED", 3: "ET"}
LABEL_REMAP  = {0: 0, 1: 1, 2: 2, 3: 3, 4: 3}  # ET(4) -> ET(3)

# ============================================================
#  REPRODUCIBILITY
# ============================================================
def set_seed(seed: int = CFG.seed) -> None:
    """Đặt seed toàn cục để tái tạo kết quả."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

# ── Device ───────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

logger.info("✅ Imports & Config OK")
logger.info(f"Device : {device}")
if torch.cuda.is_available():
    logger.info(f"GPU    : {torch.cuda.get_device_name(0)}")
    logger.info(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
logger.info(f"MONAI  : {monai.__version__} | PyTorch: {torch.__version__}")

## 3. Cấu Hình Đường Dẫn Dữ Liệu BraTS 2023

In [ ]:
# ── Mount Google Drive (Colab) ────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

# ── Tự động tìm dataset ──────────────────────────────────────
DATA_ROOT_CANDIDATES = [
    os.environ.get("BRATS_DATA_ROOT"),
    os.environ.get("DATA_ROOT"),
    "/content/drive/MyDrive/BraTS2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData",
]

DATA_ROOT  = None
all_cases: List[str] = []

for candidate in DATA_ROOT_CANDIDATES:
    if not candidate:
        continue
    path = Path(candidate)
    if path.exists():
        dirs = sorted([d.name for d in path.iterdir() if d.is_dir()])
        if dirs:
            DATA_ROOT = path
            all_cases = dirs
            break

if DATA_ROOT is None:
    DATA_ROOT = Path(DATA_ROOT_CANDIDATES[-1])
    logger.warning("Dataset không tìm thấy!")
    logger.warning("Hãy mount Google Drive và đặt đường dẫn đúng vào DATA_ROOT_CANDIDATES.")
else:
    logger.info(f"✅ Dataset tìm thấy!")
    logger.info(f"   Path  : {DATA_ROOT.resolve()}")
    logger.info(f"   Cases : {len(all_cases)} cases")
    logger.info(f"   Ví dụ : {all_cases[:3]}")

## 4. Khảo Sát Dữ Liệu (EDA)

Trước khi train, ta kiểm tra:
- Cấu trúc file: shape, spacing, dtype
- Phân bố nhãn (label distribution)
- Tính toàn vẹn dữ liệu (integrity check)
- Visualization mẫu

In [ ]:
# ── 4.1 Kiểm tra cấu trúc 1 case mẫu ────────────────────────
if all_cases:
    _case    = all_cases[0]
    _case_dir = DATA_ROOT / _case

    print(f"\n{'='*60}")
    print(f"  THÔNG TIN CASE MẪU: {_case}")
    print(f"{'='*60}")

    for mod in CFG.modalities:
        _nii = nib.load(_case_dir / f"{_case}-{mod}.nii.gz")
        _arr = _nii.get_fdata()
        _spacing = np.abs(np.diag(_nii.affine)[:3])
        print(f"  {mod.upper()}: shape={_arr.shape} | dtype={_arr.dtype} "
              f"| spacing={_spacing.round(2)} mm "
              f"| range=[{_arr.min():.1f}, {_arr.max():.1f}]")

    # Phân bố nhãn
    _seg = nib.load(_case_dir / f"{_case}-seg.nii.gz").get_fdata().astype(np.int32)
    print(f"\n  PHÂN BỐ NHÃN (raw labels): {np.unique(_seg)}")
    for v in np.unique(_seg):
        pct = 100.0 * (_seg == v).sum() / _seg.size
        name = CLASS_NAMES.get(v, f"Label {v}")
        print(f"    Label {v} ({name:12s}): {pct:.3f}% voxels")
    print(f"{'='*60}")
else:
    logger.warning("Chưa có case nào để khảo sát.")

In [ ]:
# ── 4.2 Integrity check — quét tất cả cases ──────────────────
if all_cases:
    ALL_MODS = list(CFG.modalities) + ["seg"]
    missing  = {}

    for case_id in tqdm(all_cases, desc="Integrity check"):
        case_dir = DATA_ROOT / case_id
        absent = [m for m in ALL_MODS
                  if not (case_dir / f"{case_id}-{m}.nii.gz").exists()]
        if absent:
            missing[case_id] = absent

    print(f"\n✅ Integrity check xong: {len(all_cases)} cases")
    print(f"   Cases bị thiếu file: {len(missing)}")
    if missing:
        for cid, mods in list(missing.items())[:5]:
            print(f"   ⚠️ {cid}: thiếu {mods}")
else:
    logger.warning("Chưa có case nào để kiểm tra.")

In [ ]:
# ── 4.3 Visualization — 6 cases mẫu ─────────────────────────
if all_cases:
    rng       = np.random.default_rng(CFG.seed)
    vis_cases = rng.choice(all_cases, size=min(6, len(all_cases)), replace=False)

    fig, axes = plt.subplots(len(vis_cases), 5, figsize=(20, 4 * len(vis_cases)))
    fig.suptitle("BraTS 2023 — Sample Visualization", fontsize=16, fontweight="bold", y=1.01)

    col_titles = ["T1n", "T1c", "T2w", "T2f", "Mask Overlay"]
    label_colors = {1: "red", 2: "blue", 3: "yellow"}

    for row, cid in enumerate(vis_cases):
        case_dir = DATA_ROOT / cid
        try:
            vols = {mod: nib.load(case_dir / f"{cid}-{mod}.nii.gz").get_fdata()
                    for mod in CFG.modalities}
            seg  = nib.load(case_dir / f"{cid}-seg.nii.gz").get_fdata().astype(np.int32)

            tumor_per_slice = (seg > 0).sum(axis=(0, 1))
            z = int(np.argmax(tumor_per_slice)) if tumor_per_slice.max() > 0 \
                else seg.shape[2] // 2

            for col, mod in enumerate(CFG.modalities):
                ax = axes[row, col]
                ax.imshow(np.rot90(vols[mod][:, :, z]), cmap="gray")
                ax.axis("off")
                if row == 0:
                    ax.set_title(col_titles[col], fontweight="bold", fontsize=12)

            # Mask overlay
            ax = axes[row, 4]
            ax.imshow(np.rot90(vols["t2f"][:, :, z]), cmap="gray")
            seg_slice = seg[:, :, z]
            for label_val, color in label_colors.items():
                masked = np.ma.masked_where(seg_slice != label_val, seg_slice)
                ax.imshow(np.rot90(masked), cmap=plt.cm.colors.ListedColormap([color]),
                          alpha=0.5, vmin=label_val, vmax=label_val)
            ax.axis("off")
            if row == 0:
                ax.set_title(col_titles[4], fontweight="bold", fontsize=12)
                patches = [mpatches.Patch(color=c, label=CLASS_NAMES[l])
                           for l, c in label_colors.items()]
                ax.legend(handles=patches, loc="lower right", fontsize=7)

            axes[row, 0].set_ylabel(f"{cid}\n(z={z})", fontsize=8, rotation=0,
                                     labelpad=70, va="center")
        except Exception as e:
            for col in range(5):
                axes[row, col].axis("off")
            axes[row, 2].text(0.5, 0.5, f"Lỗi:\n{e}",
                              ha="center", va="center", color="red", fontsize=9)

    plt.tight_layout()
    plt.show()
    print(f"✅ Đã hiển thị {len(vis_cases)} cases mẫu.")
else:
    logger.warning("Chưa có case nào để visualize.")

## 5. Dataset & DataLoader

**`BraTSDataset3D`** đọc 4 MRI modalities + mask, áp dụng z-score normalization theo vùng não.

In [ ]:
# ── Train / Val Split ────────────────────────────────────────
train_cases, val_cases = train_test_split(
    all_cases,
    test_size=CFG.val_split,
    random_state=CFG.seed
)
logger.info(f"✂️  Split: Train={len(train_cases)} | Val={len(val_cases)}")

In [ ]:
class BraTSDataset3D(Dataset):
    """PyTorch Dataset cho BraTS 2023 GLI.

    Đọc 4 MRI modalities + segmentation mask.
    Áp dụng z-score normalization theo vùng não (brain mask).
    Remaps label ET(4) → ET(3).

    Args:
        data_root  : Đường dẫn thư mục gốc chứa các case.
        case_ids   : Danh sách ID các case.
        transforms : MONAI Compose transforms (optional).
    """

    def __init__(self, data_root: Path, case_ids: List[str],
                 transforms=None) -> None:
        self.data_root  = Path(data_root)
        self.case_ids   = case_ids
        self.transforms = transforms

    def __len__(self) -> int:
        return len(self.case_ids)

    def _load_and_normalize(self, fpath: Path) -> np.ndarray:
        """Load NIfTI + z-score normalization theo vùng não."""
        arr  = nib.load(fpath).get_fdata().astype(np.float32)
        mask = arr > 0
        if mask.any():
            arr[mask] = (arr[mask] - arr[mask].mean()) / (arr[mask].std() + 1e-8)
        return arr

    def _remap_label(self, seg: np.ndarray) -> np.ndarray:
        """Remap raw BraTS labels: ET(4) -> ET(3)."""
        remapped = np.zeros_like(seg)
        for src, dst in LABEL_REMAP.items():
            remapped[seg == src] = dst
        return remapped

    def __getitem__(self, idx: int) -> Dict:
        case_id  = self.case_ids[idx]
        case_dir = self.data_root / case_id

        # Load & normalize 4 modalities
        channels = [
            self._load_and_normalize(case_dir / f"{case_id}-{mod}.nii.gz")
            for mod in CFG.modalities
        ]
        image = np.stack(channels, axis=0)  # (4, H, W, D)

        # Load & remap segmentation mask
        seg_path = case_dir / f"{case_id}-seg.nii.gz"
        label    = nib.load(seg_path).get_fdata().astype(np.int16)
        label    = self._remap_label(label)  # (H, W, D)

        sample = {"image": image, "label": label}
        if self.transforms:
            sample = self.transforms(sample)
        return sample

In [ ]:
# ── MONAI Transforms ─────────────────────────────────────────
train_transforms = mt.Compose([
    # Random crop 128³ — ưu tiên vùng có tumor
    mt.RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=CFG.patch_size,
        pos=1.0, neg=1.0,
        num_samples=CFG.num_samples,
        image_key="image"
    ),
    # Flip ngẫu nhiên theo 3 trục
    mt.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    mt.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    mt.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
    # Xoay 90 độ ngẫu nhiên
    mt.RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
    # Augmentation intensity
    mt.RandScaleIntensityd(keys="image", factors=0.1, prob=0.5),
    mt.RandShiftIntensityd(keys="image", offsets=0.1, prob=0.5),
    mt.RandGaussianNoised(keys="image", prob=0.2, std=0.01),
    mt.EnsureTyped(keys=["image", "label"]),
])

val_transforms = mt.Compose([
    mt.EnsureTyped(keys=["image", "label"]),
])

# ── Datasets ─────────────────────────────────────────────────
train_dataset = BraTSDataset3D(DATA_ROOT, train_cases, transforms=train_transforms)
val_dataset   = BraTSDataset3D(DATA_ROOT, val_cases,   transforms=val_transforms)

# ── DataLoaders ──────────────────────────────────────────────
train_loader = DataLoader(
    train_dataset, batch_size=CFG.batch_size, shuffle=True,
    num_workers=CFG.num_workers, pin_memory=True, persistent_workers=True
)
val_loader = DataLoader(
    val_dataset, batch_size=CFG.batch_size, shuffle=False,
    num_workers=CFG.num_workers, pin_memory=True, persistent_workers=True
)

logger.info(f"✅ DataLoaders sẵn sàng")
logger.info(f"   Train : {len(train_dataset)} cases")
logger.info(f"   Val   : {len(val_dataset)} cases")

## 6. Định Nghĩa Mô Hình — 3D U-Net

- **Input**: `(B, 4, 128, 128, 128)` — 4 MRI modalities
- **Output**: `(B, 4, 128, 128, 128)` — 4 classes (Background / NCR / ED / ET)
- **Architecture**: Encoder `[32→64→128→256→512]` + Decoder (skip connections)

In [ ]:
def build_unet() -> nn.Module:
    """Xây dựng 3D U-Net chuẩn MONAI.

    Returns:
        nn.Module: Mô hình 3D U-Net chưa được đưa lên device.
    """
    return UNet(
        spatial_dims=3,
        in_channels=CFG.in_channels,
        out_channels=CFG.out_channels,
        channels=CFG.unet_features,
        strides=CFG.unet_strides,
        num_res_units=CFG.unet_res_units,
        norm="BATCH",
    )


# ── Khởi tạo model ───────────────────────────────────────────
model = build_unet().to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

logger.info(f"✅ Model     : {model.__class__.__name__}")
logger.info(f"   Encoder  : {list(CFG.unet_features)}")
logger.info(f"   Params   : {trainable_params:,} trainable / {total_params:,} total")
logger.info(f"   Device   : {device}")

## 7. Training Loop

**Các kỹ thuật sử dụng:**
| Kỹ thuật | Mục đích |
|---|---|
| `DiceCELoss` | Xử lý class imbalance (tumor rất nhỏ ~0.6% voxel) |
| `AdamW` | Optimizer tốt hơn Adam cho medical imaging |
| `CosineAnnealingLR` | Learning rate schedule — giảm mượt mà |
| `AMP` (mixed precision) | Tăng tốc ~2x trên T4/A100, tiết kiệm VRAM |
| Gradient Clipping | Tránh gradient exploding |
| Sliding Window Inference | Inference đúng trên ảnh 240³ |
| Per-class Dice | Báo cáo NCR/ED/ET riêng biệt |

In [ ]:
# ── Checkpoint paths ─────────────────────────────────────────
CKPT_DIR  = Path(CFG.ckpt_dir);   CKPT_DIR.mkdir(exist_ok=True)
BEST_CKPT = CKPT_DIR / "unet_best.pth"
LAST_CKPT = CKPT_DIR / "unet_last.pth"
HIST_FILE = CKPT_DIR / "history.json"

# ── Loss ─────────────────────────────────────────────────────
loss_fn = DiceCELoss(to_onehot_y=True, softmax=True)

# ── Optimizer ────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.lr,
    weight_decay=CFG.weight_decay
)

# ── Scheduler ────────────────────────────────────────────────
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG.max_epochs
)

# ── AMP Scaler ───────────────────────────────────────────────
scaler = GradScaler()

# ── Metrics ──────────────────────────────────────────────────
# Per-class Dice (NCR / ED / ET)
dice_metric_per_class = DiceMetric(
    include_background=False, reduction="mean_batch"
)
# Mean Dice (trung bình 3 classes)
dice_metric_mean = DiceMetric(
    include_background=False, reduction="mean"
)

post_pred  = AsDiscrete(argmax=True, to_onehot=CFG.out_channels)
post_label = AsDiscrete(to_onehot=CFG.out_channels)

# ── History ──────────────────────────────────────────────────
history = {
    "train_loss" : [],
    "val_dice"   : [],
    "val_dice_ncr": [],
    "val_dice_ed" : [],
    "val_dice_et" : [],
    "lr"         : [],
}
best_dice = -1.0

logger.info(f"📋 Config Training:")
logger.info(f"   Epochs       : {CFG.max_epochs}")
logger.info(f"   LR           : {CFG.lr}")
logger.info(f"   Val interval : mỗi {CFG.val_interval} epoch")
logger.info(f"   AMP          : {'✅ Bật' if torch.cuda.is_available() else '❌ Tắt (không có GPU)'}")
logger.info(f"   Checkpoint   : {CKPT_DIR.resolve()}")

In [ ]:
# ============================================================
#  TRAINING LOOP CHÍNH
# ============================================================
logger.info(f"🚀 Bắt đầu training 3D U-Net trên {str(device).upper()}")
logger.info("-" * 60)

epoch_pbar = tqdm(range(1, CFG.max_epochs + 1), desc="Training", unit="epoch")

for epoch in epoch_pbar:
    # ── TRAIN PHASE ──────────────────────────────────────────
    model.train()
    epoch_loss = 0.0
    t0 = time.time()

    for batch in train_loader:
        # RandCropByPosNegLabeld với num_samples > 1 trả về list
        if isinstance(batch, list):
            images = torch.cat([b["image"] for b in batch], dim=0).to(device)
            labels = torch.cat([b["label"] for b in batch], dim=0).to(device)
        else:
            images = batch["image"].to(device)
            labels = batch["label"].to(device)

        optimizer.zero_grad()

        # AMP — Mixed Precision
        with autocast():
            outputs = model(images)
            loss    = loss_fn(outputs, labels)

        scaler.scale(loss).backward()

        # Gradient Clipping
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=CFG.grad_clip)

        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()

    epoch_loss /= len(train_loader)
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    history["train_loss"].append(epoch_loss)
    history["lr"].append(current_lr)

    dt = time.time() - t0
    epoch_pbar.set_postfix({"loss": f"{epoch_loss:.4f}", "lr": f"{current_lr:.2e}"})
    logger.info(f"Epoch {epoch:3d}/{CFG.max_epochs} | Loss: {epoch_loss:.4f} "
                f"| LR: {current_lr:.2e} | Time: {dt:.1f}s")

    # ── VALIDATION PHASE ─────────────────────────────────────
    if epoch % CFG.val_interval == 0:
        model.eval()
        t0_val = time.time()

        with torch.no_grad():
            for batch in val_loader:
                images = batch["image"].to(device)
                labels = batch["label"].to(device)

                # Sliding window inference cho ảnh 240³
                val_out = sliding_window_inference(
                    inputs=images,
                    roi_size=CFG.patch_size,
                    sw_batch_size=CFG.sw_batch_size,
                    predictor=model
                )

                preds_list  = [post_pred(i)  for i in decollate_batch(val_out)]
                labels_list = [post_label(i) for i in decollate_batch(labels)]

                dice_metric_per_class(y_pred=preds_list, y=labels_list)
                dice_metric_mean(y_pred=preds_list, y=labels_list)

        # Tổng hợp metrics
        per_class = dice_metric_per_class.aggregate()  # [ncr, ed, et]
        mean_dice = dice_metric_mean.aggregate().item()
        dice_metric_per_class.reset()
        dice_metric_mean.reset()

        dice_ncr = per_class[0].item()
        dice_ed  = per_class[1].item()
        dice_et  = per_class[2].item()

        history["val_dice"].append(mean_dice)
        history["val_dice_ncr"].append(dice_ncr)
        history["val_dice_ed"].append(dice_ed)
        history["val_dice_et"].append(dice_et)

        dt_val = time.time() - t0_val
        logger.info(f"  → Val | Mean={mean_dice:.4f} | "
                    f"NCR={dice_ncr:.4f} | ED={dice_ed:.4f} | ET={dice_et:.4f} "
                    f"| Time: {dt_val:.1f}s")

        # ── Lưu Best Checkpoint ──────────────────────────────
        if mean_dice > best_dice:
            best_dice = mean_dice
            torch.save({
                "epoch"               : epoch,
                "model_state_dict"    : model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict"   : scaler.state_dict(),
                "best_dice"           : best_dice,
                "config"              : asdict(CFG),
            }, BEST_CKPT)
            logger.info(f"  🌟 Best checkpoint lưu! Dice={best_dice:.4f} → {BEST_CKPT.name}")

        # ── Lưu Last Checkpoint ──────────────────────────────
        torch.save({
            "epoch"               : epoch,
            "model_state_dict"    : model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict"   : scaler.state_dict(),
            "best_dice"           : best_dice,
            "config"              : asdict(CFG),
        }, LAST_CKPT)

    # ── Lưu History JSON sau mỗi epoch ───────────────────────
    with open(HIST_FILE, "w") as f:
        json.dump(history, f, indent=2)

logger.info(f"✅ Training hoàn tất! Best Val Dice: {best_dice:.4f}")

## 8. Đồ Thị Learning Curves

Trực quan hóa quá trình train: Loss giảm dần & Dice score tăng dần theo epoch.

In [ ]:
# Load history từ file (đề phòng Colab crash)
if HIST_FILE.exists():
    with open(HIST_FILE) as f:
        history = json.load(f)

if len(history["train_loss"]) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    fig.suptitle("Training Progress — 3D U-Net BraTS 2023",
                 fontsize=14, fontweight="bold")

    epochs_train = range(1, len(history["train_loss"]) + 1)
    val_epochs   = [i * CFG.val_interval for i in range(1, len(history["val_dice"]) + 1)]

    # ── Plot 1: Training Loss ────────────────────────────────
    axes[0].plot(epochs_train, history["train_loss"], color="#e74c3c", lw=2)
    axes[0].set_title("Training Loss (DiceCE)", fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, linestyle="--", alpha=0.5)
    axes[0].set_xlim(1, CFG.max_epochs)

    # ── Plot 2: Validation Mean Dice ─────────────────────────
    axes[1].plot(val_epochs, history["val_dice"], color="#2ecc71",
                 lw=2, marker="o", markersize=4, label="Mean Dice")
    axes[1].axhline(max(history["val_dice"]) if history["val_dice"] else 0,
                    color="#27ae60", linestyle="--", alpha=0.7,
                    label=f"Best={max(history['val_dice']):.4f}" if history["val_dice"] else "")
    axes[1].set_title("Validation Mean Dice", fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Dice Score")
    axes[1].set_ylim(0, 1)
    axes[1].grid(True, linestyle="--", alpha=0.5)
    axes[1].legend()

    # ── Plot 3: Per-class Dice ───────────────────────────────
    colors = {"NCR": "#e74c3c", "ED": "#3498db", "ET": "#f39c12"}
    for key, label, color in [
        ("val_dice_ncr", "NCR", "#e74c3c"),
        ("val_dice_ed",  "ED",  "#3498db"),
        ("val_dice_et",  "ET",  "#f39c12"),
    ]:
        if history[key]:
            axes[2].plot(val_epochs, history[key], color=color,
                        lw=2, marker="o", markersize=4, label=label)
    axes[2].set_title("Validation Dice Per Class", fontweight="bold")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Dice Score")
    axes[2].set_ylim(0, 1)
    axes[2].grid(True, linestyle="--", alpha=0.5)
    axes[2].legend()

    plt.tight_layout()
    plt.savefig(CKPT_DIR / "learning_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ Đồ thị đã lưu vào: {CKPT_DIR / 'learning_curves.png'}")
else:
    logger.warning("Chưa có dữ liệu training để vẽ đồ thị.")

## 9. Đánh Giá Cuối Cùng

Load best checkpoint và chạy đánh giá đầy đủ trên toàn bộ validation set.

In [ ]:
if BEST_CKPT.exists():
    # ── Load best model ──────────────────────────────────────
    ckpt = torch.load(BEST_CKPT, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    logger.info(f"✅ Load best checkpoint: epoch={ckpt['epoch']} | Dice={ckpt['best_dice']:.4f}")

    # ── Đánh giá trên toàn bộ val set ───────────────────────
    dice_final_per_class = DiceMetric(include_background=False, reduction="mean_batch")
    dice_final_mean      = DiceMetric(include_background=False, reduction="mean")

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Final Evaluation"):
            images = batch["image"].to(device)
            labels = batch["label"].to(device)

            val_out = sliding_window_inference(
                inputs=images,
                roi_size=CFG.patch_size,
                sw_batch_size=CFG.sw_batch_size,
                predictor=model
            )

            preds_list  = [post_pred(i)  for i in decollate_batch(val_out)]
            labels_list = [post_label(i) for i in decollate_batch(labels)]

            dice_final_per_class(y_pred=preds_list, y=labels_list)
            dice_final_mean(y_pred=preds_list, y=labels_list)

    per_class = dice_final_per_class.aggregate()
    mean_dice = dice_final_mean.aggregate().item()

    # ── In bảng kết quả ──────────────────────────────────────
    print("\n" + "="*55)
    print("  KẾT QUẢ ĐÁNH GIÁ CUỐI CÙNG — 3D U-Net BraTS 2023")
    print("="*55)
    print(f"  {'Metric':<20} {'Dice Score':>12}")
    print("-"*55)
    print(f"  {'NCR (Necrosis)':<20} {per_class[0].item():>12.4f}")
    print(f"  {'ED (Edema)':<20} {per_class[1].item():>12.4f}")
    print(f"  {'ET (Enhancing Tumor)':<20} {per_class[2].item():>12.4f}")
    print("-"*55)
    print(f"  {'Mean Dice':<20} {mean_dice:>12.4f}")
    print("="*55)
    print(f"  Checkpoint   : {BEST_CKPT.name}")
    print(f"  Best epoch   : {ckpt['epoch']}")
    print("="*55)
else:
    logger.warning(f"Không tìm thấy checkpoint tại {BEST_CKPT}. Hãy chạy Training trước.")

## 10. Export Model (TorchScript & ONNX)

Chuyển đổi model sang:
- **TorchScript (`.pt`)**: Dùng cho C++ backend
- **ONNX (`.onnx`)**: Tối ưu cho Web Backend (FastAPI + ONNX Runtime)

In [ ]:
MODELS_DIR = Path(CFG.models_dir);  MODELS_DIR.mkdir(exist_ok=True)

TS_PATH   = MODELS_DIR / "unet.pt"
ONNX_PATH = MODELS_DIR / "unet.onnx"

if BEST_CKPT.exists():
    # Load best model
    ckpt = torch.load(BEST_CKPT, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    dummy_input = torch.randn(1, CFG.in_channels, *CFG.patch_size).to(device)

    # ── 1. Export TorchScript ────────────────────────────────
    try:
        with torch.no_grad():
            traced = torch.jit.trace(model, dummy_input)
        torch.jit.save(traced, TS_PATH)
        size_mb = os.path.getsize(TS_PATH) / 1e6
        logger.info(f"✅ TorchScript : {TS_PATH.resolve()} ({size_mb:.1f} MB)")
    except Exception as e:
        logger.error(f"❌ Lỗi export TorchScript: {e}")

    # ── 2. Export ONNX ──────────────────────────────────────
    try:
        torch.onnx.export(
            model,
            dummy_input,
            ONNX_PATH,
            export_params=True,
            opset_version=16,
            do_constant_folding=True,
            input_names=["input"],
            output_names=["output"],
            dynamic_axes={
                "input" : {0: "batch_size"},
                "output": {0: "batch_size"}
            }
        )
        size_mb = os.path.getsize(ONNX_PATH) / 1e6
        logger.info(f"✅ ONNX        : {ONNX_PATH.resolve()} ({size_mb:.1f} MB)")
    except Exception as e:
        logger.error(f"❌ Lỗi export ONNX: {e}")

    print("\n" + "="*50)
    print("  ✅ Export hoàn tất! Files đã lưu vào:")
    print(f"     {TS_PATH.resolve()}")
    print(f"     {ONNX_PATH.resolve()}")
    print("="*50)
else:
    logger.warning(f"Không tìm thấy {BEST_CKPT.name}. Hãy chạy Training trước.")